# F1 Active-Aero — Full Reproducibility Run (Kaggle GPU)

This notebook regenerates **every result** in the paper from the 4 raw circuit CSVs:

| Stage | Output | Hardware |
|---|---|---|
| 1. Build windows + LOCO scalers | `windows_H*.npz` | CPU |
| 2. Classical baselines (LR, RF-instant, RF-lag) | `baseline_results.csv` | CPU |
| 3. XGBoost + Persistence | `xgb_persistence_results.csv` | CPU |
| 4. Deep models (CNN, LSTM, GRU, TCN, Transformer) | `deep_results.csv` | **GPU** |
| 5. Feature ablation (FULL vs NO-POS) | `feature_ablation_results.csv` | CPU |
| 6. Event-level + calibration metrics | `event_metrics_results.csv` | CPU |
| 7. Integrated Gradients (GRU vs Transformer @ Monaco) | `ig_*.npy` | GPU |
| 8. Bundle everything | `f1_results_bundle.zip` | — |

## Setup on Kaggle
1. Create a **Dataset** containing the 4 files:
   `monaco_red_bull.csv`, `monza_red_bull.csv`, `silverstone_red_bull.csv`, `suzuka_red_bull.csv`
   (from `anticipatory_aero/multi_circuit_work/inputs/raw/`)
2. **Add Data** → attach that dataset to this notebook.
3. Settings → **Accelerator: GPU T4 x2** (or P100). Internet: On (for xgboost).
4. Run All. Download `f1_results_bundle.zip` from the output when finished.

Total runtime: ~30–60 min on a T4 (deep-model sweep dominates).


In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────────
import sys, subprocess
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)
pip("xgboost")              # captum optional; we implement IG manually to avoid version pin

import os, glob, pickle, random, warnings, zipfile, time
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.use_deterministic_algorithms(True, warn_only=True)
set_seed()

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────────
MASS_KG          = 798.0
SPEED_THRESHOLD  = 240.0
GEAR_THRESHOLD   = 6
W                = 50                 # window length (5 s @ 10 Hz)
HORIZONS         = [1, 5, 10, 25, 50] # full sweep
PRIMARY_H        = 10

# 12 features, order matters (X,Y,Z at indices 5,6,7 -> removed in NO-POS ablation)
FEATURE_COLS = ["Speed","RPM","nGear","Throttle","Brake",
                "X","Y","Z",
                "Acceleration","Elevation_Delta","Kinetic_Energy_MJ","Longitudinal_Force_N"]
POSITION_IDX = [5, 6, 7]
PHYSICS_IDX  = [i for i in range(len(FEATURE_COLS)) if i not in POSITION_IDX]

OUT = Path("/kaggle/working/processed"); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "deep_preds").mkdir(exist_ok=True)
(OUT / "baseline_preds").mkdir(exist_ok=True)
(OUT / "xgb_preds").mkdir(exist_ok=True)
(OUT / "models").mkdir(exist_ok=True)

# locate the 4 raw CSVs anywhere under /kaggle/input
CIRCUIT_FILES = {}
for p in glob.glob("/kaggle/input/**/*red_bull.csv", recursive=True):
    name = Path(p).stem.split("_")[0].capitalize()
    CIRCUIT_FILES[name] = p
# fallback: also accept files named exactly <circuit>.csv
print("Found circuit CSVs:")
for k, v in sorted(CIRCUIT_FILES.items()):
    print(f"  {k:12s} -> {v}")
assert len(CIRCUIT_FILES) >= 2, "Need >=2 circuit CSVs attached as a Kaggle dataset."

In [ ]:
# ── Load + clean each circuit (mirrors 01_build_anticipatory_dataset.py) ─────────
def coerce_brake(s):
    if pd.api.types.is_bool_dtype(s): return s.astype(np.float32)
    if s.dtype == object:
        return s.map({"True":1.,"False":0.,"true":1.,"false":0.,"1":1.,"0":0.}).fillna(0.).astype(np.float32)
    return pd.to_numeric(s, errors="coerce").fillna(0.).astype(np.float32)

def load_and_clean(path, circuit):
    df = pd.read_csv(path)
    df["Circuit"] = circuit
    for c in ["Speed","RPM","nGear","Throttle","X","Y","Z","LapNumber","Time_Elapsed_Sec"]:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
    df["Brake"] = coerce_brake(df["Brake"]) if "Brake" in df.columns else 0.0
    df = df.sort_values(["Driver","LapNumber","Time_Elapsed_Sec"], kind="mergesort").reset_index(drop=True)
    if "Acceleration" not in df.columns:
        df["Acceleration"] = df.groupby(["Driver","LapNumber"])["Speed"].diff().fillna(0.)
    else:
        df["Acceleration"] = pd.to_numeric(df["Acceleration"], errors="coerce").fillna(0.)
    if "Elevation_Delta" not in df.columns:
        df["Elevation_Delta"] = df.groupby(["Driver","LapNumber"])["Z"].diff().fillna(0.)
    else:
        df["Elevation_Delta"] = pd.to_numeric(df["Elevation_Delta"], errors="coerce").fillna(0.)
    sp, ac = df["Speed"].fillna(0.), df["Acceleration"].fillna(0.)
    df["Kinetic_Energy_MJ"]    = 0.5 * MASS_KG * (sp/3.6)**2 / 1e6
    df["Longitudinal_Force_N"] = MASS_KG * (ac/3.6)
    df["LapKey"] = (circuit + "|" + df["Driver"].astype(str) + "|L" +
                    df["LapNumber"].fillna(-1).astype(int).astype(str))
    df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    return df

DFS = []
for circuit, path in sorted(CIRCUIT_FILES.items()):
    d = load_and_clean(path, circuit)
    DFS.append(d)
    xrate = ((d["Speed"]>=SPEED_THRESHOLD)&(d["nGear"]>=GEAR_THRESHOLD)).mean()
    print(f"{circuit:12s}  frames={len(d):>7,}  X-Mode%={xrate*100:5.1f}")

In [ ]:
# ── Build sliding windows + LOCO scalers for every horizon ───────────────────────
from sklearn.preprocessing import StandardScaler

def build_windows(dfs, W, H):
    Xs, ys, cs, lks = [], [], [], []
    for df in dfs:
        circuit = df["Circuit"].iloc[0]
        for lap_key, lap in df.groupby("LapKey", sort=False):
            lap = lap.sort_values("Time_Elapsed_Sec").reset_index(drop=True)
            L = len(lap)
            if L < W + H: continue
            feat = lap[FEATURE_COLS].to_numpy(np.float32)
            sp   = lap["Speed"].to_numpy(np.float32)
            gr   = lap["nGear"].to_numpy(np.float32)
            for i in range(L - W - H + 1):
                j = i + W + H - 1
                ys.append(int(sp[j] >= SPEED_THRESHOLD and gr[j] >= GEAR_THRESHOLD))
                Xs.append(feat[i:i+W]); cs.append(circuit); lks.append(str(lap_key))
    return (np.stack(Xs).astype(np.float32), np.array(ys, np.int8),
            np.array(cs, object), np.array(lks, object))

WIN = {}
for H in HORIZONS:
    X, y, circ, lap = build_windows(DFS, W, H)
    WIN[H] = (X, y, circ, lap)
    np.savez_compressed(OUT/f"windows_H{H:03d}.npz", X=X, y=y, circuits=circ, lap_keys=lap)
    # LOCO scalers (fit on train windows only, flattened)
    for held in sorted(set(circ.tolist())):
        tr = circ != held
        sc = StandardScaler().fit(X[tr].reshape(tr.sum(), -1))
        with open(OUT/f"scaler_holdout_{held}_H{H:03d}.pkl","wb") as f: pickle.dump(sc, f)
    print(f"H={H:>2}  windows={X.shape}  X-Mode%={y.mean()*100:.1f}")
print("Windows + scalers built.")

In [ ]:
# ── Shared metrics ───────────────────────────────────────────────────────────────
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             matthews_corrcoef, balanced_accuracy_score, brier_score_loss)

def frame_metrics(yt, yp_prob, yp):
    npos, nneg = int(yt.sum()), int((1-yt).sum())
    return dict(
        f1_macro=float(f1_score(yt,yp,average="macro",zero_division=0)),
        f1_xmode=float(f1_score(yt,yp,pos_label=1,zero_division=0)),
        f1_zmode=float(f1_score(yt,yp,pos_label=0,zero_division=0)),
        auc_roc=float(roc_auc_score(yt,yp_prob)) if npos and nneg else float("nan"),
        auc_pr=float(average_precision_score(yt,yp_prob)) if npos else float("nan"),
        mcc=float(matthews_corrcoef(yt,yp)),
        bal_acc=float(balanced_accuracy_score(yt,yp)),
        brier=float(brier_score_loss(yt,np.clip(yp_prob,0,1))) if npos and nneg else float("nan"),
    )

In [ ]:
# ── Stage 2: Classical baselines (LR-instant, RF-instant, RF-lag) ────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def run_classical(H):
    X, y, circ, lap = WIN[H]
    rows = []
    for held in sorted(set(circ.tolist())):
        te = circ == held; tr = ~te
        ytr, yte = y[tr], y[te]
        Xi_tr, Xi_te = X[tr,-1,:], X[te,-1,:]
        sc = StandardScaler().fit(Xi_tr)
        Xi_tr_s, Xi_te_s = sc.transform(Xi_tr), sc.transform(Xi_te)
        with open(OUT/f"scaler_holdout_{held}_H{H:03d}.pkl","rb") as f: sc_lag = pickle.load(f)
        Xl_tr = sc_lag.transform(X[tr].reshape(tr.sum(),-1))
        Xl_te = sc_lag.transform(X[te].reshape(te.sum(),-1))
        specs = {
            "LR-instant": (LogisticRegression(C=1.,max_iter=2000,class_weight="balanced",
                            solver="lbfgs",random_state=SEED), Xi_tr_s, Xi_te_s),
            "RF-instant": (RandomForestClassifier(n_estimators=300,max_depth=15,
                            class_weight="balanced",n_jobs=-1,random_state=SEED), Xi_tr_s, Xi_te_s),
            "RF-lag":     (RandomForestClassifier(n_estimators=300,max_depth=15,
                            class_weight="balanced",n_jobs=-1,random_state=SEED), Xl_tr, Xl_te),
        }
        for name,(clf,Xt,Xv) in specs.items():
            clf.fit(Xt, ytr)
            prob = clf.predict_proba(Xv)[:,1]; pred = (prob>=0.5).astype(int)
            m = frame_metrics(yte, prob, pred); m.update(model=name, held_out=held, H=H)
            rows.append(m)
            np.savez_compressed(OUT/"baseline_preds"/f"{name}_{held}_H{H:03d}.npz",
                                y_prob=prob, y_true=yte, y_pred=pred)
            print(f"  H={H:>2} {name:11s} {held:11s} AUC={m['auc_roc']:.3f} F1={m['f1_macro']:.3f}")
    return rows

base_rows = []
for H in HORIZONS: base_rows += run_classical(H)
pd.DataFrame(base_rows).to_csv(OUT/"baseline_results.csv", index=False)
print("Saved baseline_results.csv")

In [ ]:
# ── Stage 3: XGBoost-instant + Persistence ───────────────────────────────────────
from xgboost import XGBClassifier

def run_xgb_persist(H):
    X, y, circ, lap = WIN[H]; rows = []
    for held in sorted(set(circ.tolist())):
        te = circ == held; tr = ~te
        ytr, yte = y[tr], y[te]
        Xi_tr, Xi_te = X[tr,-1,:], X[te,-1,:]
        sc = StandardScaler().fit(Xi_tr)
        Xi_tr_s, Xi_te_s = sc.transform(Xi_tr), sc.transform(Xi_te)
        spw = float((ytr==0).sum())/max(float((ytr==1).sum()),1)
        xgb = XGBClassifier(n_estimators=500,max_depth=6,learning_rate=0.05,subsample=0.8,
                            colsample_bytree=0.8,scale_pos_weight=spw,eval_metric="logloss",
                            random_state=SEED,n_jobs=-1,verbosity=0,
                            tree_method="hist", device="cuda" if torch.cuda.is_available() else "cpu")
        xgb.fit(Xi_tr_s, ytr)
        prob = xgb.predict_proba(Xi_te_s)[:,1]; pred=(prob>=0.5).astype(int)
        m = frame_metrics(yte, prob, pred); m.update(model="XGBoost-instant", held_out=held, H=H); rows.append(m)
        np.savez_compressed(OUT/"xgb_preds"/f"XGBoost-instant_{held}_H{H:03d}.npz", y_prob=prob,y_true=yte,y_pred=pred)
        # persistence: current-frame label as future prediction
        sp_last, gr_last = X[te,-1,0], X[te,-1,2]
        yper = ((sp_last>=SPEED_THRESHOLD)&(gr_last>=GEAR_THRESHOLD)).astype(int)
        m2 = frame_metrics(yte, yper.astype(float), yper); m2.update(model="Persistence", held_out=held, H=H); rows.append(m2)
        np.savez_compressed(OUT/"xgb_preds"/f"Persistence_{held}_H{H:03d}.npz", y_prob=yper.astype(float),y_true=yte,y_pred=yper)
        print(f"  H={H:>2} {held:11s} XGB AUC={m['auc_roc']:.3f}  Persist AUC={m2['auc_roc']:.3f}")
    return rows

xp_rows = []
for H in HORIZONS: xp_rows += run_xgb_persist(H)
pd.DataFrame(xp_rows).to_csv(OUT/"xgb_persistence_results.csv", index=False)
print("Saved xgb_persistence_results.csv")

In [ ]:
# ── Stage 4a: Deep model architectures (mirror 03_deep_models.py) ────────────────
import torch.nn as nn, torch.nn.functional as Fn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0): super().__init__(); self.a=alpha; self.g=gamma
    def forward(self, logits, t):
        bce = Fn.binary_cross_entropy_with_logits(logits, t.float(), reduction="none")
        pt = torch.exp(-bce); at = t*self.a + (1-t)*(1-self.a)
        return (at*(1-pt)**self.g*bce).mean()

class CausalConv1d(nn.Module):
    def __init__(self, ic, oc, k, d=1): super().__init__(); self.p=(k-1)*d; self.c=nn.Conv1d(ic,oc,k,dilation=d)
    def forward(self,x): return self.c(Fn.pad(x,(self.p,0)))

class CNN1D(nn.Module):
    def __init__(self, W, F, ch=64, dr=0.2):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(F,ch,5), nn.BatchNorm1d(ch), nn.ReLU(), nn.Dropout(dr),
            CausalConv1d(ch,ch*2,5), nn.BatchNorm1d(ch*2), nn.ReLU(), nn.Dropout(dr),
            CausalConv1d(ch*2,ch*2,5), nn.BatchNorm1d(ch*2), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(ch*2,ch), nn.ReLU(), nn.Dropout(dr+0.1), nn.Linear(ch,1))
    def forward(self,x): x=self.net(x.transpose(1,2)).mean(-1); return self.head(x).squeeze(-1)

class CausalLSTM(nn.Module):
    def __init__(self, F, h=128, nl=2, dr=0.2):
        super().__init__(); self.lstm=nn.LSTM(F,h,nl,batch_first=True,dropout=dr if nl>1 else 0.)
        self.head=nn.Sequential(nn.Linear(h,h//2),nn.ReLU(),nn.Dropout(dr+0.1),nn.Linear(h//2,1))
    def forward(self,x): o,_=self.lstm(x); return self.head(o[:,-1,:]).squeeze(-1)

class CausalGRU(nn.Module):
    def __init__(self, F, h=128, nl=2, dr=0.2):
        super().__init__(); self.gru=nn.GRU(F,h,nl,batch_first=True,dropout=dr if nl>1 else 0.)
        self.head=nn.Sequential(nn.Linear(h,h//2),nn.ReLU(),nn.Dropout(dr+0.1),nn.Linear(h//2,1))
    def forward(self,x): o,_=self.gru(x); return self.head(o[:,-1,:]).squeeze(-1)

class _TCNBlock(nn.Module):
    def __init__(self, ic, oc, k, d, dr):
        super().__init__(); self.c1=CausalConv1d(ic,oc,k,d); self.c2=CausalConv1d(oc,oc,k,d)
        self.b1=nn.BatchNorm1d(oc); self.b2=nn.BatchNorm1d(oc); self.dr=nn.Dropout(dr)
        self.proj=nn.Conv1d(ic,oc,1) if ic!=oc else None
    def forward(self,x):
        r=x if self.proj is None else self.proj(x)
        x=self.dr(Fn.relu(self.b1(self.c1(x)))); x=self.dr(Fn.relu(self.b2(self.c2(x))))
        return Fn.relu(x+r)

class TCN(nn.Module):
    def __init__(self, F, ch=64, k=5, dr=0.2):
        super().__init__(); ic=F; L=[]
        for d in [1,2,4,8]: L.append(_TCNBlock(ic,ch,k,d,dr)); ic=ch
        self.net=nn.Sequential(*L)
        self.head=nn.Sequential(nn.Linear(ch,ch//2),nn.ReLU(),nn.Dropout(dr+0.1),nn.Linear(ch//2,1))
    def forward(self,x): x=self.net(x.transpose(1,2)); return self.head(x[:,:,-1]).squeeze(-1)

class _PE(nn.Module):
    def __init__(self, d, ml=200):
        super().__init__(); pe=torch.zeros(ml,d); pos=torch.arange(ml).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,d,2).float()*(-np.log(10000.)/d))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div); self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class CausalTransformer(nn.Module):
    def __init__(self, F, dm=64, nh=4, nl=2, ff=128, dr=0.1):
        super().__init__(); self.proj=nn.Linear(F,dm); self.pe=_PE(dm)
        el=nn.TransformerEncoderLayer(dm,nh,ff,dr,batch_first=True)
        self.enc=nn.TransformerEncoder(el,nl); self.head=nn.Sequential(nn.LayerNorm(dm),nn.Linear(dm,1))
    @staticmethod
    def _mask(n,dev): m=torch.triu(torch.ones(n,n,device=dev),1); return m.masked_fill(m==1,float("-inf"))
    def forward(self,x):
        x=self.pe(self.proj(x)); m=self._mask(x.size(1),x.device)
        x=self.enc(x,mask=m,is_causal=True); return self.head(x[:,-1,:]).squeeze(-1)

def build_model(name, W, F):
    return {"CNN":CNN1D(W,F),"LSTM":CausalLSTM(F),"GRU":CausalGRU(F),
            "TCN":TCN(F),"Transformer":CausalTransformer(F)}[name]
DEEP_MODELS = ["CNN","LSTM","GRU","TCN","Transformer"]
print("Deep model defs ready:", DEEP_MODELS)

In [ ]:
# ── Stage 4b: Train all deep models under LOCO (GPU) ─────────────────────────────
MAX_EPOCHS, PATIENCE, BATCH = 80, 12, 512
DEEP_HORIZONS = [PRIMARY_H]   # set to HORIZONS for the full horizon sweep (slower)

def make_ds(X, y, scaler):
    N,W_,F_ = X.shape
    Xs = scaler.transform(X.reshape(N,-1)).reshape(N,W_,F_).astype(np.float32)
    return TensorDataset(torch.from_numpy(Xs), torch.from_numpy(y.astype(np.float32)))

@torch.no_grad()
def eval_loader(model, loader):
    model.eval(); P,Y=[],[]
    for xb,yb in loader:
        P.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy()); Y.append(yb.numpy())
    return np.concatenate(P), np.concatenate(Y)

def fit(model, tr_ds, va_ds, crit):
    set_seed()
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=0.5,patience=5,min_lr=1e-5)
    trl=DataLoader(tr_ds,BATCH,shuffle=True); val=DataLoader(va_ds,BATCH*2,shuffle=False)
    best, best_state, bad = 0., None, 0
    for ep in range(1, MAX_EPOCHS+1):
        model.train()
        for xb,yb in trl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            loss=crit(model(xb),yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        vp,vt=eval_loader(model,val)
        ap=average_precision_score(vt,vp) if 0<vt.sum()<len(vt) else 0.5
        sch.step(ap)
        if ap>best+1e-5: best,best_state,bad=ap,{k:v.detach().clone() for k,v in model.state_dict().items()},0
        else: bad+=1
        if bad>=PATIENCE: break
    if best_state: model.load_state_dict(best_state)
    return model, ep

deep_rows=[]; t0=time.time()
for H in DEEP_HORIZONS:
    X,y,circ,lap = WIN[H]; N,Wd,Fd = X.shape
    for held in sorted(set(circ.tolist())):
        te=circ==held; tr_idx=np.where(~te)[0]; ytr_all=y[~te]
        with open(OUT/f"scaler_holdout_{held}_H{H:03d}.pkl","rb") as f: scaler=pickle.load(f)
        try: idx_tr,idx_va=train_test_split(tr_idx,test_size=0.1,stratify=ytr_all,random_state=SEED)
        except ValueError: idx_tr,idx_va=train_test_split(tr_idx,test_size=0.1,random_state=SEED)
        tr_ds=make_ds(X[idx_tr],y[idx_tr],scaler); va_ds=make_ds(X[idx_va],y[idx_va],scaler)
        te_ds=make_ds(X[te],y[te],scaler); te_loader=DataLoader(te_ds,BATCH*2,shuffle=False)
        alpha=1.0-float(y[idx_tr].mean()); crit=FocalLoss(alpha=alpha,gamma=2.0).to(DEVICE)
        for name in DEEP_MODELS:
            set_seed(); model=build_model(name,Wd,Fd).to(DEVICE)
            model,ep=fit(model,tr_ds,va_ds,crit)
            prob,yte=eval_loader(model,te_loader); pred=(prob>=0.5).astype(int)
            m=frame_metrics(yte,prob,pred); m.update(model=name,held_out=held,H=H,stopped_epoch=ep)
            deep_rows.append(m)
            np.savez_compressed(OUT/"deep_preds"/f"{name}_{held}_H{H:03d}.npz",y_prob=prob,y_true=yte,y_pred=pred)
            torch.save({"model_state":model.state_dict(),"W":Wd,"F":Fd,"H":H,"held_out":held},
                       OUT/"models"/f"{name}_fold_{held}_H{H:03d}.pt")
            print(f"  H={H} {name:11s} {held:11s} ep={ep:>2} AUC={m['auc_roc']:.3f} F1={m['f1_macro']:.3f}")
pd.DataFrame(deep_rows).to_csv(OUT/"deep_results.csv", index=False)
print(f"Saved deep_results.csv  ({time.time()-t0:.0f}s)")

In [ ]:
# ── Stage 5: Feature ablation FULL vs NO-POS (LR + RF) ───────────────────────────
def run_ablation(H):
    X,y,circ,lap = WIN[H]; rows=[]
    conds={"FULL":list(range(len(FEATURE_COLS))),"NO-POS":PHYSICS_IDX}
    for held in sorted(set(circ.tolist())):
        te=circ==held; tr=~te; ytr,yte=y[tr],y[te]
        for cond,idx in conds.items():
            Xt=X[tr,-1,:][:,idx]; Xv=X[te,-1,:][:,idx]
            sc=StandardScaler().fit(Xt); Xt,Xv=sc.transform(Xt),sc.transform(Xv)
            for name,clf in [("LR-instant",LogisticRegression(C=1.,max_iter=2000,class_weight="balanced",solver="lbfgs",random_state=SEED)),
                             ("RF-instant",RandomForestClassifier(n_estimators=300,max_depth=15,class_weight="balanced",n_jobs=-1,random_state=SEED))]:
                clf.fit(Xt,ytr); prob=clf.predict_proba(Xv)[:,1]; pred=(prob>=0.5).astype(int)
                m=frame_metrics(yte,prob,pred); m.update(model=name,condition=cond,held_out=held,H=H,n_feat=len(idx))
                rows.append(m)
    return rows
abl=pd.DataFrame(run_ablation(PRIMARY_H)); abl.to_csv(OUT/"feature_ablation_results.csv",index=False)
piv=abl[abl.H==PRIMARY_H].groupby(["model","condition"])["auc_roc"].mean().unstack()
piv["drop"]=(piv["FULL"]-piv["NO-POS"]).round(4); print(piv.round(4).to_string())
print("Saved feature_ablation_results.csv")

In [ ]:
# ── Stage 6: Event-level switch metrics + calibration ────────────────────────────
def find_switches(lab, lk):
    sw=[]
    for L in np.unique(lk):
        idx=np.where(lk==L)[0]
        if len(idx)<2: continue
        ch=np.where(lab[idx][1:]!=lab[idx][:-1])[0]+1
        sw += [idx[c] for c in ch]
    return sorted(sw)

def event_metrics(yt, yp, lk, tol=5):
    ts=find_switches(yt,lk); ps=np.array(find_switches(yp,lk))
    if not ts: return dict(detection_rate=np.nan,mean_timing_err=np.nan,false_toggle_rate=np.nan,n_true_switches=0)
    matched=set(); errs=[]; det=0
    for t in ts:
        if len(ps)==0: continue
        d=ps-t; w=np.where(np.abs(d)<=tol)[0]
        if len(w): k=w[np.argmin(np.abs(d[w]))]; det+=1; errs.append(int(d[k])); matched.add(int(ps[k]))
    errs=np.array(errs) if errs else np.array([0])
    return dict(detection_rate=det/len(ts), mean_timing_err=float(errs.mean()),
                false_toggle_rate=(len(ps)-len(matched))/max(len(ts),1), n_true_switches=len(ts))

def lapkeys(held,H):
    d=np.load(OUT/f"windows_H{H:03d}.npz",allow_pickle=True)
    return d["lap_keys"][d["circuits"]==held]

PRED_DIRS=[OUT/"baseline_preds",OUT/"xgb_preds",OUT/"deep_preds"]
def find_pred(model,held,H):
    for d in PRED_DIRS:
        f=d/f"{model}_{held}_H{H:03d}.npz"
        if f.exists(): return f
    return None

CIRCS=sorted(set(WIN[PRIMARY_H][2].tolist()))
all_models=["LR-instant","RF-instant","XGBoost-instant","RF-lag","Persistence"]+DEEP_MODELS
ev_rows=[]
for model in all_models:
    for held in CIRCS:
        f=find_pred(model,held,PRIMARY_H)
        if f is None: continue
        d=np.load(f,allow_pickle=True); yt=d["y_true"].astype(int); pr=d["y_prob"].astype(float); yp=d["y_pred"].astype(int)
        lk=lapkeys(held,PRIMARY_H)
        ev=event_metrics(yt,yp,lk) if len(lk)==len(yt) else dict(detection_rate=np.nan,mean_timing_err=np.nan,false_toggle_rate=np.nan,n_true_switches=0)
        row=dict(model=model,held_out=held,H=PRIMARY_H); row.update(frame_metrics(yt,pr,yp)); row.update(ev); ev_rows.append(row)
ev=pd.DataFrame(ev_rows); ev.to_csv(OUT/"event_metrics_results.csv",index=False)
cols=["auc_roc","auc_pr","mcc","brier","detection_rate","mean_timing_err","false_toggle_rate"]
print(ev.groupby("model")[cols].mean().round(3).to_string())
print("Saved event_metrics_results.csv")

In [ ]:
# ── Stage 7: Integrated Gradients (GRU vs Transformer @ Monaco) ──────────────────
# NOTE: cuDNN's fused RNN kernel cannot run its backward pass in eval() mode, which
# breaks gradient-based attribution for GRU/LSTM. We disable cuDNN for the duration of
# the IG computation so PyTorch falls back to the native RNN kernel (which supports
# eval-mode backward). model.eval() is kept so dropout/BatchNorm stay deterministic.
def integrated_gradients(model, x, baseline=None, steps=50):
    model.eval()
    if baseline is None: baseline = torch.zeros_like(x)
    total = torch.zeros_like(x)
    cudnn_was = torch.backends.cudnn.enabled
    torch.backends.cudnn.enabled = False           # native RNN kernel -> eval-mode backward OK
    try:
        with torch.enable_grad():
            for a in np.linspace(0, 1, steps):
                xi = (baseline + a * (x - baseline)).clone().requires_grad_(True)
                out = model(xi).sum()
                grad, = torch.autograd.grad(out, xi)
                total += grad.detach()
    finally:
        torch.backends.cudnn.enabled = cudnn_was   # restore for the rest of the notebook
    return ((x - baseline) * total / steps).detach()

IG_TARGET = "Monaco" if "Monaco" in CIRCS else CIRCS[0]
ig_summary = {}
for name in ["GRU","Transformer"]:
    ckpt = OUT/"models"/f"{name}_fold_{IG_TARGET}_H{PRIMARY_H:03d}.pt"
    if not ckpt.exists(): print(f"skip IG {name}: no checkpoint"); continue
    st = torch.load(ckpt, map_location=DEVICE)
    model = build_model(name, st["W"], st["F"]).to(DEVICE); model.load_state_dict(st["model_state"])
    X,y,circ,lap = WIN[PRIMARY_H]; te = circ==IG_TARGET
    with open(OUT/f"scaler_holdout_{IG_TARGET}_H{PRIMARY_H:03d}.pkl","rb") as f: scaler=pickle.load(f)
    Xs = scaler.transform(X[te].reshape(te.sum(),-1)).reshape(te.sum(),st["W"],st["F"]).astype(np.float32)
    sub = torch.from_numpy(Xs[:512]).to(DEVICE)        # sample for tractability
    attr = integrated_gradients(model, sub).abs().mean(dim=(0,1)).cpu().numpy()  # per-feature
    np.save(OUT/f"ig_{name}_{IG_TARGET}_H{PRIMARY_H:03d}.npy", attr)
    ig_summary[name] = dict(zip(FEATURE_COLS, attr.round(4)))
    top = sorted(zip(FEATURE_COLS,attr), key=lambda z:-z[1])[:4]
    print(f"{name} @ {IG_TARGET} top features:", [(f,round(float(v),3)) for f,v in top])
pd.DataFrame(ig_summary).to_csv(OUT/"ig_feature_attributions.csv")
print("Saved IG attributions.")

In [ ]:
# ── Stage 8: Assemble paper-ready summary tables ─────────────────────────────────
def summ(df, H=PRIMARY_H):
    d=df[df.H==H]
    return d.groupby("model")[["auc_roc","auc_pr","f1_macro","f1_xmode"]].agg(["mean","std"]).round(3)

base=pd.read_csv(OUT/"baseline_results.csv")
xp=pd.read_csv(OUT/"xgb_persistence_results.csv")
deep=pd.read_csv(OUT/"deep_results.csv")
allres=pd.concat([base,xp,deep],ignore_index=True)
allres.to_csv(OUT/"ALL_results_combined.csv",index=False)

print("="*70); print("MAIN RESULTS @ H=10 (mean across LOCO folds)"); print("="*70)
main=allres[allres.H==PRIMARY_H].groupby("model")[["auc_roc","auc_pr","f1_macro","f1_xmode"]].mean().round(3)
main=main.sort_values("auc_roc",ascending=False); print(main.to_string())
main.to_csv(OUT/"TABLE_main_results_H10.csv")
print("\\nSaved TABLE_main_results_H10.csv + ALL_results_combined.csv")

In [ ]:
# ── Stage 9: Bundle everything into a single downloadable zip ─────────────────────
bundle = Path("/kaggle/working/f1_results_bundle.zip")
keep_ext = {".csv",".npy",".pkl",".pt"}
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and (p.suffix in keep_ext or "preds" in str(p)):
            z.write(p, p.relative_to(OUT.parent))
size_mb = bundle.stat().st_size/1e6
print(f"Bundle ready: {bundle}  ({size_mb:.1f} MB)")
print("Download it from the Kaggle output panel (right sidebar) when the run finishes.")
print("\\nContents summary:")
for f in sorted(OUT.glob('*.csv')): print("  ", f.name)